# USDA Fruit and Vegetable Price Explorer

This notebook connects the database work from the previous assignment to Python data analysis and interactive visualization.

You will use three files:

1. `fruit_vegetable_prices.db`, your SQLite database containing `fruit_prices` and `vegetable_prices`
2. The USDA `ALL FRUITS - Average prices` CSV
3. The USDA `ALL VEGETABLES - Average prices` CSV

USDA Fruit and Vegetable Prices:
https://www.ers.usda.gov/data-products/fruit-and-vegetable-prices

USDA Highlights and Interactive Charts:
https://www.ers.usda.gov/data-products/fruit-and-vegetable-prices/highlights-and-interactive-charts

**Important:** USDA cautions that the annual datasets should not be used to infer price changes over time. This notebook compares products within the current USDA dataset and uses the older SQLite data for database querying, classification, and integration.

## 1. Setup

Run this cell first. The notebook uses:

- `sqlite3` to query the SQLite database
- `pandas` for data manipulation
- `plotly` for interactive charts
- `ipywidgets` for interactive controls inside Google Colab

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets

from IPython.display import display, Markdown
from google.colab import files, output

output.enable_custom_widget_manager()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Setup complete.")

## 2. Upload the three data files

Upload your SQLite database plus the two CSV files downloaded from USDA.

You do not need to rename the USDA files. The code below will identify the fruit and vegetable CSV files by examining their columns.

In [ ]:
uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(" -", filename)

## 3. Find and inspect the SQLite database

This section demonstrates that the data from the previous MariaDB assignment can be moved into a portable SQLite database and queried directly from Python.

In [ ]:
db_files = [f for f in uploaded if f.lower().endswith((".db", ".sqlite", ".sqlite3"))]

if len(db_files) == 0:
    raise FileNotFoundError("No SQLite database file was uploaded.")

db_file = db_files[0]
print("Using SQLite database:", db_file)

conn = sqlite3.connect(db_file)

tables = pd.read_sql_query(
    "SELECT name AS table_name FROM sqlite_master WHERE type = 'table' ORDER BY name;",
    conn
)

display(tables)

### Query the original 2020 tables

These queries are executed by SQLite. Pandas receives the query results after the database has processed the SQL.

In [ ]:
fruit_2020 = pd.read_sql_query(
    "SELECT * FROM fruit_prices;",
    conn
)

vegetable_2020 = pd.read_sql_query(
    "SELECT * FROM vegetable_prices;",
    conn
)

print(f"fruit_prices rows: {len(fruit_2020):,}")
print(f"vegetable_prices rows: {len(vegetable_2020):,}")

display(fruit_2020.head())
display(vegetable_2020.head())

### Example database summary query

This is an additional SQL query against the SQLite database. It summarizes the number of rows stored in each table.

In [ ]:
table_counts = pd.read_sql_query(
    """
    SELECT 'fruit_prices' AS table_name, COUNT(*) AS row_count
    FROM fruit_prices

    UNION ALL

    SELECT 'vegetable_prices' AS table_name, COUNT(*) AS row_count
    FROM vegetable_prices;
    """,
    conn
)

display(table_counts)

## 4. Retrieve the legume classification from SQLite

The previous assignment added a `Legume` field to `vegetable_prices`. This cell retrieves the foods classified as legumes.

The code checks the table structure first so the notebook will still run if the column name uses different capitalization.

In [ ]:
veg_columns = pd.read_sql_query("PRAGMA table_info(vegetable_prices);", conn)
display(veg_columns[["name", "type"]])

column_lookup = {c.lower(): c for c in vegetable_2020.columns}

veg_name_col_2020 = column_lookup.get("vegetable")
legume_col = column_lookup.get("legume")

if veg_name_col_2020 and legume_col:
    legume_query = f'''
        SELECT DISTINCT "{veg_name_col_2020}" AS Vegetable
        FROM vegetable_prices
        WHERE UPPER(TRIM("{legume_col}")) = 'Y'
        ORDER BY "{veg_name_col_2020}";
    '''
    legumes_2020 = pd.read_sql_query(legume_query, conn)
    print(f"Distinct legume entries found in SQLite: {len(legumes_2020):,}")
    display(legumes_2020)
else:
    legumes_2020 = pd.DataFrame(columns=["Vegetable"])
    print("A Vegetable and/or Legume column was not found. The rest of the notebook will still run.")

## 5. Load the current USDA fruit and vegetable CSV files

The current USDA files use these fields:

- `Fruit` or `Vegetable`
- `Form`
- `RetailPrice`
- `RetailPriceUnit`
- `Yield`
- `CupEquivalentSize`
- `CupEquivalentUnit`
- `CupEquivalentPrice`

The next cell automatically identifies which uploaded CSV is the fruit file and which is the vegetable file.

In [ ]:
csv_files = [f for f in uploaded if f.lower().endswith(".csv")]

if len(csv_files) < 2:
    raise FileNotFoundError("Please upload both USDA CSV files.")

fruit_csv_file = None
vegetable_csv_file = None
fruit_current = None
vegetable_current = None

for filename in csv_files:
    test_df = pd.read_csv(filename)
    cols_lower = {c.lower(): c for c in test_df.columns}

    if "fruit" in cols_lower:
        fruit_csv_file = filename
        fruit_current = test_df
    elif "vegetable" in cols_lower:
        vegetable_csv_file = filename
        vegetable_current = test_df

if fruit_current is None:
    raise ValueError("Could not identify the USDA fruit CSV. Expected a column named Fruit.")

if vegetable_current is None:
    raise ValueError("Could not identify the USDA vegetable CSV. Expected a column named Vegetable.")

print("Fruit CSV:", fruit_csv_file)
print("Vegetable CSV:", vegetable_csv_file)
print(f"Fruit records: {len(fruit_current):,}")
print(f"Vegetable records: {len(vegetable_current):,}")

## 6. Explore the USDA data

Before creating charts, inspect the structure and quality of both datasets.

In [ ]:
print("FRUIT COLUMNS")
print(fruit_current.columns.tolist())

print("\nVEGETABLE COLUMNS")
print(vegetable_current.columns.tolist())

print("\nFRUIT SAMPLE")
display(fruit_current.head())

print("\nVEGETABLE SAMPLE")
display(vegetable_current.head())

In [ ]:
print("FRUIT DATA TYPES")
display(fruit_current.dtypes.rename("dtype").to_frame())

print("\nVEGETABLE DATA TYPES")
display(vegetable_current.dtypes.rename("dtype").to_frame())

print("\nFRUIT MISSING VALUES")
display(fruit_current.isna().sum().rename("missing_values").to_frame())

print("\nVEGETABLE MISSING VALUES")
display(vegetable_current.isna().sum().rename("missing_values").to_frame())

### Questions to answer

Add your answers below this cell.

1. What does one row in the USDA dataset represent?
2. Which column identifies the fruit or vegetable?
3. Which column identifies how the product is sold or prepared?
4. Which column represents its retail price?
5. Which column represents the cost of one edible cup equivalent?
6. Why might price per cup equivalent be more useful for comparing foods than retail price alone?

### Student answers

1. 
2. 
3. 
4. 
5. 
6. 

## 7. Combine fruits and vegetables

The fruit and vegetable CSV files use different names for the food column. We will standardize both to `Item`, add a `Category` column, and combine them into one DataFrame.

In [ ]:
fruit = fruit_current.rename(columns={"Fruit": "Item"}).copy()
fruit["Category"] = "Fruit"

vegetable = vegetable_current.rename(columns={"Vegetable": "Item"}).copy()
vegetable["Category"] = "Vegetable"

prices = pd.concat([fruit, vegetable], ignore_index=True)

numeric_columns = [
    "RetailPrice",
    "Yield",
    "CupEquivalentSize",
    "CupEquivalentPrice"
]

for col in numeric_columns:
    if col in prices.columns:
        prices[col] = pd.to_numeric(prices[col], errors="coerce")

prices = prices.dropna(subset=["Item", "Form", "CupEquivalentPrice"]).copy()

print(f"Combined records: {len(prices):,}")
display(prices.head(10))

In [ ]:
print("Records by category:")
display(
    prices.groupby("Category")
          .size()
          .rename("records")
          .reset_index()
)

print("\nRecords by product form:")
display(
    prices.groupby(["Category", "Form"])
          .size()
          .rename("records")
          .reset_index()
          .sort_values(["Category", "records"], ascending=[True, False])
)

## 8. Interactive Visualization 1: Cost per cup equivalent

Use the controls to explore fruits, vegetables, product forms, and the number of products shown.

The chart ranks products by their estimated cost per edible cup equivalent.

In [ ]:
category_options = ["All"] + sorted(prices["Category"].dropna().unique().tolist())
form_options = ["All"] + sorted(prices["Form"].dropna().unique().tolist())

category_widget = widgets.Dropdown(
    options=category_options,
    value="All",
    description="Category:"
)

form_widget = widgets.Dropdown(
    options=form_options,
    value="All",
    description="Form:"
)

top_widget = widgets.IntSlider(
    value=20,
    min=5,
    max=min(50, len(prices)),
    step=5,
    description="Items:"
)

sort_widget = widgets.Dropdown(
    options=["Least expensive", "Most expensive"],
    value="Least expensive",
    description="Order:"
)

def cost_per_cup_chart(category, form, items, order):
    d = prices.copy()

    if category != "All":
        d = d[d["Category"] == category]

    if form != "All":
        d = d[d["Form"] == form]

    ascending = order == "Least expensive"

    d = (
        d.sort_values("CupEquivalentPrice", ascending=ascending)
         .head(items)
         .copy()
    )

    d["Label"] = d["Item"] + " (" + d["Form"] + ")"

    fig = px.bar(
        d,
        x="CupEquivalentPrice",
        y="Label",
        orientation="h",
        color="Category",
        hover_data=["Item", "Category", "Form", "RetailPrice", "RetailPriceUnit"],
        labels={
            "CupEquivalentPrice": "Cost per cup equivalent ($)",
            "Label": "Product"
        },
        title="USDA Fruit and Vegetable Cost per Cup Equivalent"
    )

    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        height=max(500, 28 * len(d))
    )

    fig.show()

controls = widgets.VBox([
    widgets.HBox([category_widget, form_widget]),
    widgets.HBox([sort_widget, top_widget])
])

out = widgets.interactive_output(
    cost_per_cup_chart,
    {
        "category": category_widget,
        "form": form_widget,
        "items": top_widget,
        "order": sort_widget
    }
)

display(controls, out)

## 9. Interactive Visualization 2: Most and least expensive products

This visualization focuses on the extremes of the USDA price distribution.

In [ ]:
extreme_category = widgets.Dropdown(
    options=category_options,
    value="All",
    description="Category:"
)

extreme_type = widgets.ToggleButtons(
    options=["Least expensive", "Most expensive"],
    value="Least expensive",
    description="Show:"
)

extreme_n = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=1,
    description="Top N:"
)

def extreme_chart(category, extreme, n):
    d = prices.copy()

    if category != "All":
        d = d[d["Category"] == category]

    ascending = extreme == "Least expensive"

    d = (
        d.sort_values("CupEquivalentPrice", ascending=ascending)
         .head(n)
         .copy()
    )

    d["Label"] = d["Item"] + " | " + d["Form"]

    fig = px.bar(
        d,
        x="CupEquivalentPrice",
        y="Label",
        orientation="h",
        color="Category",
        hover_data=["Item", "Form", "RetailPrice", "RetailPriceUnit"],
        labels={
            "CupEquivalentPrice": "Cost per cup equivalent ($)",
            "Label": "Product"
        },
        title=f"{extreme}: {category} products"
    )

    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        height=max(450, 35 * len(d))
    )

    fig.show()

extreme_out = widgets.interactive_output(
    extreme_chart,
    {
        "category": extreme_category,
        "extreme": extreme_type,
        "n": extreme_n
    }
)

display(
    widgets.HBox([extreme_category, extreme_type, extreme_n]),
    extreme_out
)

## 10. Interactive Visualization 3: Distribution of prices by form

A box plot helps compare the distribution of cost per cup equivalent across fresh, frozen, canned, dried, juice, and other product forms.

In [ ]:
box_category = widgets.Dropdown(
    options=category_options,
    value="All",
    description="Category:"
)

def distribution_chart(category):
    d = prices.copy()

    if category != "All":
        d = d[d["Category"] == category]

    form_order = (
        d.groupby("Form")["CupEquivalentPrice"]
         .median()
         .sort_values()
         .index
         .tolist()
    )

    fig = px.box(
        d,
        x="Form",
        y="CupEquivalentPrice",
        color="Category",
        points="all",
        hover_data=["Item"],
        category_orders={"Form": form_order},
        labels={
            "CupEquivalentPrice": "Cost per cup equivalent ($)",
            "Form": "Product form"
        },
        title=f"Distribution of Cost per Cup Equivalent by Form: {category}"
    )

    fig.update_layout(height=600)
    fig.show()

box_out = widgets.interactive_output(
    distribution_chart,
    {"category": box_category}
)

display(box_category, box_out)

### Interpretation

Does one product form appear to be consistently less or more expensive than another?

Write your interpretation here:

**Answer:** 

## 11. Additional analysis: Legumes versus other vegetables

This section connects the current USDA data back to the `Legume` field created in the previous database assignment.

The legume names are retrieved from SQLite, normalized, and used to classify records in the current USDA vegetable data.

In [ ]:
def normalize_food_name(value):
    return (
        str(value)
        .strip()
        .lower()
        .replace("&", "and")
        .replace("-", " ")
    )

legume_names = set()

if not legumes_2020.empty:
    legume_names = {
        normalize_food_name(v)
        for v in legumes_2020["Vegetable"].dropna()
    }

def is_legume_current(item):
    item_normalized = normalize_food_name(item)

    if not legume_names:
        return False

    return any(
        legume_name == item_normalized
        or legume_name in item_normalized
        or item_normalized in legume_name
        for legume_name in legume_names
    )

vegetable_analysis = prices[prices["Category"] == "Vegetable"].copy()
vegetable_analysis["Legume"] = np.where(
    vegetable_analysis["Item"].apply(is_legume_current),
    "Legume",
    "Other vegetable"
)

print("Classification counts:")
display(
    vegetable_analysis.groupby("Legume")
                      .size()
                      .rename("records")
                      .reset_index()
)

display(
    vegetable_analysis[
        vegetable_analysis["Legume"] == "Legume"
    ][["Item", "Form", "CupEquivalentPrice"]]
    .sort_values(["Item", "Form"])
)

In [ ]:
fig = px.box(
    vegetable_analysis,
    x="Legume",
    y="CupEquivalentPrice",
    color="Legume",
    points="all",
    hover_data=["Item", "Form"],
    labels={
        "Legume": "Vegetable classification",
        "CupEquivalentPrice": "Cost per cup equivalent ($)"
    },
    title="Cost per Cup Equivalent: Legumes versus Other Vegetables"
)

fig.update_layout(showlegend=False, height=600)
fig.show()

### Legume analysis

What question were you trying to answer, and what did you find?

**Question:** How does the cost per cup equivalent of legumes compare with other vegetables?

**Finding:** Write your interpretation here.

## 12. Optional additional view: Retail price versus cost per cup

Retail price and cost per edible cup equivalent measure different things. This scatter plot lets you examine the relationship between them.

In [ ]:
scatter_data = prices.dropna(subset=["RetailPrice", "CupEquivalentPrice"]).copy()

fig = px.scatter(
    scatter_data,
    x="RetailPrice",
    y="CupEquivalentPrice",
    color="Category",
    symbol="Form",
    hover_name="Item",
    hover_data=["Form", "RetailPriceUnit"],
    labels={
        "RetailPrice": "Retail price ($)",
        "CupEquivalentPrice": "Cost per cup equivalent ($)"
    },
    title="Retail Price versus Cost per Cup Equivalent"
)

fig.update_layout(height=650)
fig.show()

## 13. Summary tables for your final findings

Use these results to help identify patterns worth discussing. Do not simply copy the tables into your final answer. Explain what the results mean for a nontechnical reader.

In [ ]:
cheapest = prices.nsmallest(10, "CupEquivalentPrice")[
    ["Item", "Category", "Form", "CupEquivalentPrice"]
]

most_expensive = prices.nlargest(10, "CupEquivalentPrice")[
    ["Item", "Category", "Form", "CupEquivalentPrice"]
]

median_by_form = (
    prices.groupby(["Category", "Form"], as_index=False)
          .agg(
              products=("Item", "count"),
              median_cost_per_cup=("CupEquivalentPrice", "median"),
              average_cost_per_cup=("CupEquivalentPrice", "mean")
          )
          .sort_values(["Category", "median_cost_per_cup"])
)

under_50_cents = (
    prices.assign(under_50=prices["CupEquivalentPrice"] <= 0.50)
          .groupby("Category", as_index=False)
          .agg(
              total_products=("Item", "count"),
              products_at_or_below_50_cents=("under_50", "sum")
          )
)

print("10 least expensive products")
display(cheapest)

print("\n10 most expensive products")
display(most_expensive)

print("\nMedian and average cost per cup by category and form")
display(median_by_form)

print("\nProducts costing $0.50 or less per cup equivalent")
display(under_50_cents)

## 14. Key Findings

Write three findings supported by your analysis. Each finding should be understandable to a nontechnical audience.

### Finding 1

Write your first finding here.

### Finding 2

Write your second finding here.

### Finding 3

Write your third finding here.

## Data limitation

The USDA ERS documentation states that the annual Fruit and Vegetable Prices datasets should not be used to infer price changes over time because changes in product coding, market products, and methodology can affect comparisons between years. For that reason, this notebook uses the current USDA dataset for price comparisons and uses the older SQLite tables for database integration and classification.

## 15. Close the database connection

Run this cell when you are finished.

In [ ]:
conn.close()
print("SQLite connection closed.")